In [5]:
from datasets import load_dataset, Dataset, DatasetDict
import random
import numpy as np
import torch
import pandas as pd
import math
from transformers import AutoTokenizer
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    pipeline,
    AutoModelForMaskedLM
)
from transformers import DataCollatorForLanguageModeling
from transformers import DataCollatorForWholeWordMask
from datasets import concatenate_datasets

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1. Обучим NER-модель



## Загрузим набор данных gusevski/factrueval2016

In [6]:
ds = load_dataset("gusevski/factrueval2016")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train_data.json:   0%|          | 0.00/7.62M [00:00<?, ?B/s]

dev_data.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

test_data.json:   0%|          | 0.00/2.57M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1 [00:00<?, ? examples/s]

In [7]:
random.seed(77)
np.random.seed(77)
torch.manual_seed(77)
torch.cuda.manual_seed_all(77)

## Подготовим набор данных для обучения (токенизация, выравнинивание меток и т.п.)

In [8]:
label_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}

In [9]:
train = ds["train"][0]["data"]
val = ds["validation"][0]["data"]
test= ds["test"][0]["data"]

train_dataset = Dataset.from_list(train)
val_dataset = Dataset.from_list(val)
test_dataset = Dataset.from_list(test)

In [10]:
ds = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset,
})

ds

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 2582
    })
})

Возьмём модель для дообучения

In [11]:
model_name = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [12]:
def tokenize_and_align_labels(examples, max_length=256):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=max_length
    )

    aligned_labels = []
    for i in range(len(examples["tokens"])):
        word_ids = tokenized.word_ids(batch_index=i)
        gold = examples["ner_tags"][i]
        prev = None
        out = []
        for w in word_ids:
            if w is None:
                out.append(-100)
            elif w != prev:
                out.append(gold[w])
            else:
                out.append(-100)
            prev = w
        aligned_labels.append(out)

    tokenized["labels"] = aligned_labels
    return tokenized

max_length = 192
tokenized = ds.map(lambda x: tokenize_and_align_labels(x, max_length=max_length), batched=True)
tokenized

Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2582
    })
})

In [13]:
seqeval_metric = evaluate.load("seqeval")

In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    y_true, y_pred = [], []
    for p, l in zip(preds, labels):
        true_labels = []
        pred_labels = []
        for pi, li in zip(p, l):
            if li == -100:
                continue
            true_labels.append(id2label[li])
            pred_labels.append(id2label[pi])
        y_true.append(true_labels)
        y_pred.append(pred_labels)

    res = seqeval_metric.compute(predictions=y_pred, references=y_true, zero_division=0)
    return {
        "f1": res["overall_f1"],
        "precision": res["overall_precision"],
        "recall": res["overall_recall"],
    }

In [15]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding="longest")

In [16]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                        

## Дообучим подходящую энкодерную модель на train-части корпуса для решения NER-задачи, сделаем замеры качества NER-метрик до и после дообучения


Будем сохранять лучшую модель по метрике качества f1 и ориентироваться на неё, которая сбалансированно учитывает Precision и Recall, смотреть логи по шагам, а не эпохам, learning rate довольно стандартнывй для решения NER задачи

In [ ]:
training_args = TrainingArguments(
    output_dir="./baseline",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    fp16=True,
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=50,
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    seed=42,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

До обучения

In [ ]:
trainer.evaluate()

{'eval_loss': 2.0561466217041016,
 'eval_model_preparation_time': 0.011,
 'eval_f1': 0.03333179233507466,
 'eval_precision': 0.01894675986755663,
 'eval_recall': 0.13844086021505375,
 'eval_runtime': 6.8626,
 'eval_samples_per_second': 376.241,
 'eval_steps_per_second': 11.803}

Обучим

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Model Preparation Time,F1,Precision,Recall
100,0.011967,0.019890,0.011000,0.979478,0.978352,0.980607
200,0.008539,0.026704,0.011000,0.976829,0.978240,0.975422
300,0.007198,0.022073,0.011000,0.980756,0.978041,0.983487
400,0.006399,0.025151,0.011000,0.982399,0.984197,0.980607


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1100, training_loss=0.005864526747979901, metrics={'train_runtime': 1318.4405, 'train_samples_per_second': 58.751, 'train_steps_per_second': 1.843, 'total_flos': 995438660204736.0, 'train_loss': 0.005864526747979901, 'epoch': 4.527835051546392})

In [ ]:
trainer.save_model("best_modelFT")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

После обучения

In [ ]:
finetun_one = trainer.evaluate()
finetun_one

{'eval_loss': 0.023247752338647842,
 'eval_model_preparation_time': 0.011,
 'eval_f1': 0.9837920782583677,
 'eval_precision': 0.9827553171105575,
 'eval_recall': 0.9848310291858678,
 'eval_runtime': 5.3817,
 'eval_samples_per_second': 479.773,
 'eval_steps_per_second': 15.051,
 'epoch': 4.527835051546392}

Получили довольно высокое качество f1 = 0.9838

#2. Попробуем улучшить качество модели следующими способами (из оригинального чекпойнта модели с HF):

## Предварительно дообучим на train-части в MLM режиме, а потом дообучим на NER-задачу


In [ ]:
MLM_model = AutoModelForMaskedLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be igno

In [ ]:
def preprocess_function(examples):
    return tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True
    )

In [ ]:
tokenized_data = ds.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=ds["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/7746 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2582 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2582 [00:00<?, ? examples/s]

In [ ]:
tokenized_data

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1026
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 339
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 350
    })
})

In [ ]:
block_size = 192
def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [ ]:
tokenized_data = tokenized_data.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/7746 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2582 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2582 [00:00<?, ? examples/s]

In [ ]:
MLM_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm_probability=0.15
)

In [ ]:
training_args_MLM = TrainingArguments(
    output_dir="MLM_training",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=5,
    weight_decay=0.01,
    seed=42,
    load_best_model_at_end=True
)
trainer_MLM = Trainer(
    model=MLM_model,
    args=training_args_MLM,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    data_collator=MLM_collator,
)


In [ ]:
trainer_MLM.train()

Epoch,Training Loss,Validation Loss
1,No log,1.907299
2,No log,1.932465
3,No log,1.879232
4,0.833506,1.784708
5,0.833506,1.802600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=645, training_loss=0.9893427412639293, metrics={'train_runtime': 1200.2942, 'train_samples_per_second': 4.274, 'train_steps_per_second': 0.537, 'total_flos': 1050160321543680.0, 'train_loss': 0.9893427412639293, 'epoch': 5.0})

In [ ]:
trainer_MLM.save_model("best_modelMLM")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 1.02


Получили перплексию = 1.02, которая говорит из какого количества вероятных токенов выбирает наша модель, наша перплексия довольно низкая, что говорит о хорошем результате

Дообучим на NER задачу с теми же гиперпараметрами, которые были изначально при обучении просто модели на train части

In [ ]:
model_MLM_NER = AutoModelForTokenClassification.from_pretrained(
    "MLM_training/checkpoint-516/",
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: MLM_training/checkpoint-516/
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
    output_dir="./MLN_NER_training",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    fp16=True,
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=50,
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    seed=42,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

trainer_MLM_NER = Trainer(
    model=model_MLM_NER,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
    data_collator = data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
trainer_MLM_NER.train()

Step,Training Loss,Validation Loss,F1,Precision,Recall
100,0.130655,0.039570,0.933483,0.911453,0.956605
200,0.064833,0.025207,0.960305,0.954476,0.966206
300,0.032408,0.024088,0.971259,0.966002,0.976575
400,0.034320,0.019821,0.973555,0.975149,0.971966
500,0.031398,0.019042,0.976172,0.976923,0.975422
600,0.013570,0.019147,0.977061,0.976780,0.977343
700,0.019858,0.021000,0.974290,0.969933,0.978687
800,0.011662,0.018452,0.980478,0.977299,0.983679
900,0.007921,0.019103,0.981034,0.978784,0.983295
1000,0.007864,0.017980,0.980276,0.977655,0.982911


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1500, training_loss=0.05724322332938512, metrics={'train_runtime': 1519.6052, 'train_samples_per_second': 50.974, 'train_steps_per_second': 1.599, 'total_flos': 1359557860203048.0, 'train_loss': 0.05724322332938512, 'epoch': 6.17319587628866})

In [ ]:
trainer_MLM_NER.evaluate()

{'eval_loss': 0.019132383167743683,
 'eval_f1': 0.9814832581790272,
 'eval_precision': 0.9808245445829339,
 'eval_recall': 0.9821428571428571,
 'eval_runtime': 7.1126,
 'eval_samples_per_second': 363.018,
 'eval_steps_per_second': 11.388,
 'epoch': 6.17319587628866}

In [ ]:
trainer_MLM_NER.save_model("best_modelMLM_NER")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Для улучшения потенциального качества MLM попробуем не стандартное случайное маскирование отдельных токенов, а использование DataCollatorForWholeWordMask - маскирование целых слов

In [ ]:
def tokenize_wwm(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=192
    )

    tokenized["word_ids"] = [
        tokenized.word_ids(batch_index=i)
        for i in range(len(examples["tokens"]))
    ]

    return tokenized

In [ ]:
tokenized_WORD = ds.map(
    tokenize_wwm,
    batched=True,
    remove_columns=ds["train"].column_names
)

Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

In [ ]:
class DataCollatorForWWM:
    def __init__(self, tokenizer, mlm_probability=0.15):
        self.tokenizer = tokenizer
        self.mlm_probability = mlm_probability

    def __call__(self, features)
        word_ids_batch = [f["word_ids"] for f in features]

        features_for_padding = [
            {k: v for k, v in f.items() if k != "word_ids"}
            for f in features
        ]

        batch = self.tokenizer.pad(
            features_for_padding,
            return_tensors="pt"
        )

        input_ids = batch["input_ids"]
        labels = input_ids.clone()

        for i, word_ids in enumerate(word_ids_batch):

            word_to_tokens = {}
            for idx, word_id in enumerate(word_ids):
                if word_id is None or idx >= input_ids.shape[1]:
                    continue
                word_to_tokens.setdefault(word_id, []).append(idx)

            for token_indices in word_to_tokens.values():
                if random.random() < self.mlm_probability:
                    for idx in token_indices:
                        labels[i, idx] = input_ids[i, idx]

                        rand = random.random()
                        if rand < 0.8:
                            input_ids[i, idx] = self.tokenizer.mask_token_id
                        elif rand < 0.9:
                            input_ids[i, idx] = random.randint(0, len(self.tokenizer) - 1)
                else:
                    for idx in token_indices:
                        labels[i, idx] = -100
        labels[input_ids == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": batch["attention_mask"],
            "labels": labels,
        }

In [ ]:
MLMWord_collator= DataCollatorForWWM(tokenizer)

In [ ]:
training_args_MLMWord = TrainingArguments(
    output_dir="MLMWord_training",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=5,
    weight_decay=0.01,
    seed=42,
    load_best_model_at_end=True,
     remove_unused_columns=False
)
trainer_MLMWord = Trainer(
    model=MLM_model,
    args=training_args_MLMWord,
    train_dataset=tokenized_WORD["train"],
    eval_dataset=tokenized_WORD["validation"],
    data_collator=MLMWord_collator,
)


In [ ]:
tokenized_WORD

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 2582
    })
})

In [ ]:
trainer_MLMWord.train()

Epoch,Training Loss,Validation Loss
1,1.557556,1.480330
2,1.334662,1.385329
3,1.234420,1.397290
4,1.186771,1.377747
5,1.186638,1.386661


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=4845, training_loss=1.285247109868824, metrics={'train_runtime': 1976.9448, 'train_samples_per_second': 19.591, 'train_steps_per_second': 2.451, 'total_flos': 1966105769796504.0, 'train_loss': 1.285247109868824, 'epoch': 5.0})

In [ ]:
trainer_MLMWord.save_model("best_modelMLMWord")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
eval_results = trainer_MLMWord.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 4.17


Получили перплексию = 4, которая говорит из какого количества вероятных токенов выбирает наша модель, наша перплексия довольно низкая, но в сравнении с обычным маскированием получили результат более хуже

Дообучим на NER-задачу

In [ ]:
model_MLMWord_NER = AutoModelForTokenClassification.from_pretrained(
    "MLMWord_training/checkpoint-3876/",
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: MLMWord_training/checkpoint-3876/
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
    output_dir="./MLMWord_NER_training",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    fp16=True,
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=50,
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    seed=42,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

trainer_MLMWord_NER = Trainer(
    model=model_MLMWord_NER,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
    data_collator = data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
trainer_MLMWord_NER.train()

Step,Training Loss,Validation Loss,F1,Precision,Recall
100,0.115241,0.037806,0.943045,0.923601,0.963326
200,0.056305,0.027248,0.961056,0.950940,0.971390
300,0.028781,0.020152,0.974271,0.967083,0.981567
400,0.033089,0.021281,0.974433,0.975558,0.973310
500,0.027385,0.016673,0.976405,0.975470,0.977343
600,0.012287,0.017702,0.976236,0.974369,0.978111
700,0.017146,0.019546,0.977566,0.972090,0.983103
800,0.010911,0.018951,0.979767,0.974004,0.985599
900,0.008985,0.018507,0.978870,0.974862,0.982911
1000,0.009372,0.018038,0.980662,0.977854,0.983487


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1800, training_loss=0.043553903727895686, metrics={'train_runtime': 733.1862, 'train_samples_per_second': 105.648, 'train_steps_per_second': 3.314, 'total_flos': 1626011961959484.0, 'train_loss': 0.043553903727895686, 'epoch': 7.408247422680413})

In [ ]:
trainer_MLMWord_NER.evaluate()

{'eval_loss': 0.019440550357103348,
 'eval_f1': 0.983631664592706,
 'eval_precision': 0.9807215117388814,
 'eval_recall': 0.9865591397849462,
 'eval_runtime': 7.7793,
 'eval_samples_per_second': 331.906,
 'eval_steps_per_second': 10.412,
 'epoch': 7.408247422680413}

In [ ]:
trainer_MLMWord_NER.save_model("best_modelMLMWord_NER")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Видим, что мы получили качество метрики F1 чуть лучше, чем с обычным маскированием. WholeWordMask более эффективен, чем стандартный MLM

## Сгенерируем синтетическую разметку подходящего корпуса большой и умной моделью для русскоязычного NER, а затем используем ее для дообучения энкодера вместе с основным набором данных

Возьмём данные lenta ru с предыдущих домашних работ

In [29]:
df = pd.read_csv('lenta_ru_news_lemma_csv')


In [30]:
df.head(5)

,title,text,topic,full_text,token_text
0,Столица Таджикистана осталась без света,Столица Таджикистана Душанбе практически полно...,Мир,Столица Таджикистана осталась без света Столиц...,столица таджикистан остаться свет столица тадж...
1,Мотопарад в Москве установил рекорд,Более пяти тысяч человек приняли участие в мот...,Россия,Мотопарад в Москве установил рекорд Более пяти...,мотопарад москва установить рекорд пять тыся...
2,КНДР провела неудачный запуск ракеты в день ро...,Северная Корея попыталась запустить баллистиче...,Силовые структуры,КНДР провела неудачный запуск ракеты в день ро...,кндр провести неудачный запуск ракета день р...
3,Путин поручил рассмотреть вопрос о введении ут...,Президент России Владимир Путин поручил правит...,Бизнес,Путин поручил рассмотреть вопрос о введении ут...,путин поручить рассмотреть вопрос введение у...
4,Поляки отправили в Россию последнюю партию отр...,Последняя партия ядерных отходов из польского ...,Бизнес,Поляки отправили в Россию последнюю партию отр...,поляк отправить россия последний партия отра...


Возьмём модель Gherman/bert-base-NER-Russian для получения синтетической разметки на новостных данных

In [31]:
model_name_NER = "Gherman/bert-base-NER-Russian"
tokenizer_ner = AutoTokenizer.from_pretrained(model_name_NER)
model_ner = AutoModelForTokenClassification.from_pretrained(model_name_NER)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [32]:
def predict_ner(text):
    inputs = tokenizer_ner(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=192
    )

    with torch.no_grad():
        outputs = model_ner(**inputs)

    logits = outputs.logits
    preds = torch.argmax(logits, dim=-1)

    tokens = tokenizer_ner.convert_ids_to_tokens(inputs["input_ids"][0])
    labels = [model_ner.config.id2label[p.item()] for p in preds[0]]

    return tokens, labels
def merge_tokens(tokens, labels):
    words = []
    word_labels = []

    current_word = ""
    current_label = None

    for token, label in zip(tokens, labels):

        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        if "[UNK]" in token:
            continue
        if token.startswith("##"):
            current_word += token[2:]
        else:
            if current_word:
                words.append(current_word)
                word_labels.append(current_label)

            current_word = token
            current_label = label

    if current_word:
        words.append(current_word)
        word_labels.append(current_label)

    return words, word_labels

def map_label(label):
    if label == "O":
        return "O"

    if "-" in label:
        prefix, tag = label.split("-", 1)
    else:
        return "O"

    if tag in ["FIRST_NAME", "LAST_NAME"]:
        ner = "PER"
    elif tag in ["CITY", "COUNTRY", "STATE"]:
        ner = "LOC"
    elif tag in ["ORG", "ORGANIZATION"]:
        ner = "ORG"
    else:
        return "O"

    if prefix in ["U", "B"]:
        return f"B-{ner}"
    elif prefix in ["I", "L"]:
        return f"I-{ner}"

    return "O"

In [33]:
def generate_synthetic(texts, label2id):
    synthetic_data = []

    for text in texts:

        tokens, labels = predict_ner(text)

        words, word_labels = merge_tokens(tokens, labels)

        final_labels_str = [
            map_label(l)
            for l in word_labels
        ]


        final_labels = [
            label2id.get(l, label2id["O"])
            for l in final_labels_str
        ]


        synthetic_data.append({
            "tokens": words,
            "ner_tags": final_labels,
            "ner_tags_str": final_labels_str
        })

    return synthetic_data

Возьмём 2000 текстов и проведём на них синтетическую разметку

In [36]:
texts = df["full_text"].tolist()


texts = texts[:2000]

In [37]:
synthetic_data = generate_synthetic(texts, label2id)

In [38]:
synthetic_dataset = Dataset.from_list(synthetic_data)

In [39]:
synthetic_dataset

Dataset({
    features: ['tokens', 'ner_tags', 'ner_tags_str'],
    num_rows: 2000
})

Объединим данные

In [40]:
combined_dataset = concatenate_datasets([
    ds["train"],
    synthetic_dataset
])

In [41]:
combined_tokenized = combined_dataset.map(lambda x: tokenize_and_align_labels(x, max_length=max_length), batched=True)

Map:   0%|          | 0/9746 [00:00<?, ? examples/s]

In [42]:
training_args = TrainingArguments(
    output_dir="./synthetic_NER",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    fp16=True,
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=50,
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    seed=42,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

trainer_NER_synt = Trainer(
    model=model,
    args=training_args,
    train_dataset=combined_tokenized,
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

Начнём обучение на NER задачу

In [43]:
trainer_NER_synt.train()

Step,Training Loss,Validation Loss,F1,Precision,Recall
100,0.138552,0.052686,0.924521,0.913263,0.936060
200,0.066088,0.030065,0.949059,0.939594,0.958717
300,0.053416,0.025945,0.952561,0.939660,0.965822
400,0.033803,0.024744,0.968039,0.967761,0.968318
500,0.035714,0.018574,0.971936,0.966401,0.977535
600,0.033682,0.019365,0.971900,0.970876,0.972926
700,0.022915,0.017002,0.975661,0.970013,0.981375
800,0.020014,0.016645,0.976129,0.974727,0.977535
900,0.025738,0.017604,0.975283,0.973231,0.977343
1000,0.014308,0.019228,0.977795,0.979018,0.976575


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1700, training_loss=0.05249954107929679, metrics={'train_runtime': 1817.1827, 'train_samples_per_second': 53.632, 'train_steps_per_second': 1.678, 'total_flos': 3451486062143328.0, 'train_loss': 0.05249954107929679, 'epoch': 5.573770491803279})

In [44]:
trainer_NER_synt.save_model("best_modelNER_SYNT")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [45]:
trainer_NER_synt.evaluate()

{'eval_loss': 0.01812927983701229,
 'eval_f1': 0.980806142034549,
 'eval_precision': 0.9804297774366846,
 'eval_recall': 0.9811827956989247,
 'eval_runtime': 4.3817,
 'eval_samples_per_second': 589.27,
 'eval_steps_per_second': 18.486,
 'epoch': 5.573770491803279}

# Финально сравним результаты различных подходов

Метрику качества F1, на котору мы ориентируемся в данной задаче,удалось получить следующую в каждом из подходов:

1. При дообучении модели rubert на train части датасета f1 = 0.98379
2. При обучении train-части на MLM и дообучение на NER f1 = 0.98148
3. При обучении train-части на WholeWordMask и дообучениа на NER f1 = 0.98363
4. При генерации синтетической разметки и обучение общего датасета f1 = 0.98080

Везде мы получили относительно высокое качество решаемой задачи.Разница между лучшим и худшим результатом: 0.00299
Прямое дообучение на размеченных данных осталось самым эффективным подходом.
На обучение вместе с синтетической разметкой и новостными данными lenta-ru мы получили самое низкое качество, что может говорить о том, что качество разметки может быть недостаточно высоким, либо синтетические данные вносят шум

Для ещё более лучшего получения качества решаемой задачи можно предложить использовать ансамбль моделей/подходов и получать общее предсказание, использовать ещё более крупные, лучшие модели, подобрать оптимальные гиперпараметры, провести анализ ошибок и выявить типы сущностей, на которых модель ошибается больше всего